# Exploring Baselines

In [ ]:
from tok_preparer.src.settings import tmp_db

In [ ]:
import plotly.offline as pyo
import plotly.graph_objs as go

from collections import defaultdict
from itertools import pairwise

import sqlite3

In [ ]:
if not tmp_db.exists():
    raise FileNotFoundError(f"Database file not found at {tmp_db}. Please run the data preparation script first.")

In [ ]:
def get_baseline():
    with sqlite3.connect(tmp_db) as conn:
        cur = conn.cursor()

        return {x:y for x,y in cur.execute('select year, count(*) from utterance group by year').fetchall()}
baseline = get_baseline()

In [ ]:
def make_line(timeline, name):
    x,y = zip(*timeline)
    total = sum(y)
    name_with_count = f'{name} ({total:,})'
    yy = [y1/baseline[x1] for x1,y1 in timeline]
    assert len(y) == len(yy)
    return go.Scatter(mode='lines', x=x, y=y, name=name_with_count), go.Scatter(mode='lines', x=x, y=yy, name=name_with_count)


In [ ]:
pyo.init_notebook_mode(connected=False)

In [ ]:
def enable_plotly_in_cell():

  import IPython
  from plotly.offline import init_notebook_mode
  display(IPython.core.display.HTML('''<script src="/static/components/requirejs/require.js"></script>'''))
  init_notebook_mode(connected=False)


## Establishing baselines


In [ ]:
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    line, r = make_line(cur.execute('select year, count(*) from utterance group by year order by year').fetchall(), 'total')


pyo.iplot({'data' : [line,],
            'layout':{
                'title':{
                    'text': 'Number of utterances per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

In [ ]:
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    line, r = make_line(cur.execute('select year, sum(length(uf.content)) from utterance as u join utterance_fts as uf on u.rowid == uf.rowid group by year order by year').fetchall(), 'total')

pyo.iplot({'data' : [line],
            'layout':{
                'title':{
                    'text': 'Number of characters per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

In [ ]:
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    line, r = make_line(cur.execute(
        '''
        select
          year,
          sum(
            length(uf.content) -
            length(
              replace(
                replace(
                  replace(
                    uf.content,
                    " ",
                    ""
                  ),
                  "\t",
                  ""
                ),
              "\n",
              ""
              )
            )
          )
        from
          utterance as u
        join
          utterance_fts as uf
          on
            u.rowid == uf.rowid
        group by
          year
        order by
          year
        ''').fetchall(), 'total')


pyo.iplot({'data' : [line,],
            'layout':{
                'title':{
                    'text': 'Number of words per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

In [ ]:
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    line, r = make_line(cur.execute('select year, count(distinct who) from utterance group by year order by year').fetchall(), 'total')


pyo.iplot({'data' : [line,],
            'layout':{
                'title':{
                    'text': 'Number of speakers per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

In [ ]:
from itertools import pairwise
from collections import defaultdict

def speakers():
  with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    mapper = defaultdict(int)

    for (year_old, who_old), (year_new, who_new) in pairwise([(None, None)] + list(cur.execute('select year, who from utterance group by year, id order by date').fetchall())):
      if who_old != who_new or year_old != year_new:
        mapper[year_new] += 1

  return sorted(mapper.items())



line, r = make_line(speakers(), 'total')

pyo.iplot({'data' : [line,],
            'layout':{
                'title':{
                    'text': 'Number of cohesive utterances per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})


In [ ]:
def speechy():
  with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()

    for year, utt in cur.execute('select year, uf.content from utterance as u join utterance_fts as uf on u.rowid == uf.rowid where who != "unknown"').fetchall():
      yield year, utt


def utt_lens():

    mapper = defaultdict(list)

    for (year, utt) in list(speechy()):
      utt_len = len(utt.split())
      mapper[year].append(utt_len)

    mins = {year: min(lengths) for year, lengths in mapper.items()}
    maxs = {year: max(lengths) for year, lengths in mapper.items()}
    avgs = {year: sum(lengths)/len(lengths) for year, lengths in mapper.items()}
    stds = {year: (sum((l - avgs[year])**2 for l in lengths)/len(lengths))**0.5 for year, lengths in mapper.items()}
    sdt1 = {year: avgs[year] + stds[year] for year in mapper.keys()}
    sdt2 = {year: avgs[year] - stds[year] for year in mapper.keys()}
    return sorted(avgs.items()), sorted(mins.items()), sorted(maxs.items()), sorted(sdt1.items()), sorted(sdt2.items())


avgs, mins, maxs, std1, std2 = utt_lens()
avgl, r = make_line(avgs, 'mean')
minl, r = make_line(mins, 'mins')
maxl, r = make_line(maxs, 'max')
st1l, r = make_line(std1, 'std+1')
st2l, r = make_line(std2, 'std-1')


pyo.iplot({'data' : [avgl, minl, maxl, #st1l,
                      st2l],
            'layout':{
                'title':{
                    'text': 'Number of words per utterance per year',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

pyo.iplot({'data' : [avgl, minl, maxl, #st1l,
                     st2l],
            'layout':{
                'title':{
                    'text': 'Number of words per utterance per year (log scale)',
                    'xanchor': 'center',
                    'x':0.5
                },
                 "yaxis": {
                    'type': 'log',}}})

In [ ]:
import pandas as pd


def make_lark():
  lark = defaultdict(int)
  yc = defaultdict(int)

  for yr, utt in speechy():
    utt_len = len(utt.split())
    lark[(yr, utt_len)] += 1



  for (yr, utt_len), count in lark.items():
    yield {'x':yr, 'y':utt_len, 'z':count}

df = pd.DataFrame.from_records(list(make_lark()))

In [ ]:
import plotly.express as px
fig = px.density_heatmap(df, x="x", y="y", marginal_x="histogram", marginal_y="histogram")
fig.show()
